In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

# --- Parameters ---
pca_windows = [None, 756, 504, 252]
pca_start_date = '2016-06-01'
zscore_windows = [252, 180, 60, 20, 10]
z_threshold = 1.5
holding_periods = [5, 10, 20, 40, 60]
dv01 = 10000
trade_types = ['outright', 'spread_hedged', 'pure_residual']
target_country = 'PE'
target_tenor = '10Y'

# --- Spread construction ---
country_map = {
    'df_perugb_cmt': 'PE',
    'df_coltes_cmt': 'CO',
    'df_mbono_cmt': 'MX',
    'df_btpcl_cmt': 'CL',
    'df_bntnf_cmt': 'BR'
}
dfs = {
    'df_perugb_cmt': df_perugb_cmt,
    'df_coltes_cmt': df_coltes_cmt,
    'df_mbono_cmt': df_mbono_cmt,
    'df_btpcl_cmt': df_btpcl_cmt,
    'df_bntnf_cmt': df_bntnf_cmt
}
ust = df_ust_cmt.set_index('Fecha')[target_tenor].rename('UST')
spread_series = {}
for df_name, col in country_map.items():
    lc = dfs[df_name].set_index('Fecha')[target_tenor]
    spread_series[col] = (lc - ust) * 100
df_spreads = pd.DataFrame(spread_series).dropna()
df_spreads.index.name = 'Fecha'
df_spreads = df_spreads.reset_index()
df_spreads = df_spreads.dropna()
df_spreads = df_spreads.set_index('Fecha')
countries = ['PE', 'CO', 'MX', 'CL', 'BR']

# --- Daily spread changes ---
df_delta = df_spreads[countries].diff().dropna()
df_delta = df_delta.loc[pca_start_date:]
df_spreads_aligned = df_spreads.loc[df_delta.index]

# --- PCA helpers ---
def run_pca_full(delta_mat):
    sc = StandardScaler(with_mean=True, with_std=True)
    X = sc.fit_transform(delta_mat)
    pca = PCA(n_components=5)
    pca.fit(X)
    scores = pca.transform(X)
    loadings = pca.components_.T * sc.scale_
    evr = pca.explained_variance_ratio_
    return scores, loadings, evr

def rolling_ols(y, X, window):
    n = len(y)
    alpha = np.full(n, np.nan)
    beta = np.full(n, np.nan)
    resid = np.full(n, np.nan)
    for i in range(window - 1, n):
        s = max(0, i - window + 1)
        xi = X[s:i+1].reshape(-1, 1)
        yi = y[s:i+1]
        mask = ~(np.isnan(xi.ravel()) | np.isnan(yi))
        if mask.sum() < 10:
            continue
        lr = LinearRegression().fit(xi[mask], yi[mask])
        alpha[i] = lr.intercept_
        beta[i] = lr.coef_[0]
        resid[i] = yi[-1] - (lr.intercept_ + lr.coef_[0] * xi[-1, 0])
    return alpha, beta, resid

def expanding_ols(y, X, min_obs=60):
    n = len(y)
    alpha = np.full(n, np.nan)
    beta = np.full(n, np.nan)
    resid = np.full(n, np.nan)
    for i in range(min_obs, n):
        xi = X[:i+1].reshape(-1, 1)
        yi = y[:i+1]
        mask = ~(np.isnan(xi.ravel()) | np.isnan(yi))
        if mask.sum() < min_obs:
            continue
        lr = LinearRegression().fit(xi[mask], yi[mask])
        alpha[i] = lr.intercept_
        beta[i] = lr.coef_[0]
        resid[i] = y[i] - (lr.intercept_ + lr.coef_[0] * X[i])
    return alpha, beta, resid

def compute_zscore(resid_arr, window):
    s = pd.Series(resid_arr)
    mu = s.rolling(window, min_periods=window).mean()
    sg = s.rolling(window, min_periods=window).std()
    return ((s - mu) / sg).values

# --- Precompute residuals for all (pca_window, zscore_window) combos ---
# Store: residuals[pca_w][z_w] = dict with arrays indexed by df_delta.index
idx = df_delta.index
n = len(idx)
delta_mat_full = df_delta[countries].values
spread_pe = df_spreads_aligned['PE'].values
spread_all = df_spreads_aligned[countries].values
yield_pe_raw = df_perugb_cmt.set_index('Fecha')[target_tenor].reindex(idx).values

# Full-sample PCA scores, loadings, evr (for diagnostics)
scores_full, loadings_full, evr_full = run_pca_full(delta_mat_full)
pc1_scores_full = scores_full[:, 0]
pc2_scores_full = scores_full[:, 1]
pc3_scores_full = scores_full[:, 2]

# Storage
residuals_store = {}
zscore_store = {}
pc1_store = {}
loadings_store = {}

for pca_w in pca_windows:
    if pca_w is None:
        # Full-sample/expanding PCA
        pc1_arr = pc1_scores_full.copy()
        F_t = np.nancumsum(pc1_arr)
        _, _, resid_pe = expanding_ols(spread_pe, F_t, min_obs=60)
        loadings_store[pca_w] = loadings_full[:, :3]  # 5 countries x 3 PCs
    else:
        # Rolling PCA
        pc1_arr = np.full(n, np.nan)
        roll_loadings = np.full((n, len(countries)), np.nan)
        for i in range(pca_w - 1, n):
            s = max(0, i - pca_w + 1)
            sub = delta_mat_full[s:i+1]
            if sub.shape[0] < 30:
                continue
            sc = StandardScaler(with_mean=True, with_std=True)
            X_sc = sc.fit_transform(sub)
            pca_tmp = PCA(n_components=1)
            pca_tmp.fit(X_sc)
            score_i = pca_tmp.transform(X_sc[-1:, :])[0, 0]
            pc1_arr[i] = score_i
            roll_loadings[i] = (pca_tmp.components_[0] * sc.scale_)
        F_t = np.nancumsum(np.where(np.isnan(pc1_arr), 0, pc1_arr))
        F_t[np.isnan(pc1_arr)] = np.nan
        _, _, resid_pe = rolling_ols(spread_pe, F_t, window=pca_w)
        loadings_store[pca_w] = roll_loadings

    pc1_store[pca_w] = pc1_arr
    residuals_store[pca_w] = resid_pe

    zscore_store[pca_w] = {}
    for z_w in zscore_windows:
        zscore_store[pca_w][z_w] = compute_zscore(resid_pe, z_w)

# --- Per-country residuals for RV dashboard (using full-sample PCA) ---
country_residuals = {}
country_fitted = {}
best_pca_w = None
best_z_w = 60
F_best = np.nancumsum(pc1_store[best_pca_w])

for ci, c in enumerate(countries):
    sp_c = df_spreads_aligned[c].values
    _, _, resid_c = expanding_ols(sp_c, F_best, min_obs=60)
    country_residuals[c] = resid_c
    fitted_c = sp_c - resid_c
    country_fitted[c] = fitted_c

# --- df_rv_today ---
last_i = n - 1
rv_rows = []
for c in countries:
    resid_arr = country_residuals[c]
    resid_series = pd.Series(resid_arr)
    mu = resid_series.rolling(best_z_w, min_periods=best_z_w).mean()
    sg = resid_series.rolling(best_z_w, min_periods=best_z_w).std()
    z_arr = ((resid_series - mu) / sg).values
    sp_last = df_spreads_aligned[c].iloc[-1]
    fit_last = country_fitted[c][-1]
    res_last = resid_arr[-1]
    z_last = z_arr[-1]
    if z_last > z_threshold:
        sig = 'CHEAP'
    elif z_last < -z_threshold:
        sig = 'RICH'
    else:
        sig = '—'
    rv_rows.append({'Country': c, 'Spread (bps)': round(sp_last, 1),
                    'Fitted (bps)': round(fit_last, 1),
                    'Residual (bps)': round(res_last, 1),
                    'Z-score': round(z_last, 2), 'Signal': sig})
df_rv_today = pd.DataFrame(rv_rows)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from itertools import product

# ============================================================
# BACKTEST ENGINE
# ============================================================
def run_backtest(pca_w, z_w, hp, trade_type):
    z_arr = zscore_store[pca_w][z_w]
    resid_arr = residuals_store[pca_w]
    sp_pe = df_spreads_aligned['PE'].values
    yld_pe = yield_pe_raw

    trades = []
    i = 0
    while i < n - hp:
        z = z_arr[i]
        if np.isnan(z):
            i += 1
            continue
        if z > z_threshold:
            direction = 1  # long (cheap)
        elif z < -z_threshold:
            direction = -1  # short (rich)
        else:
            i += 1
            continue
        j = min(i + hp, n - 1)
        if trade_type == 'outright':
            y0 = yld_pe[i]
            y1 = yld_pe[j]
            if np.isnan(y0) or np.isnan(y1):
                i += hp
                continue
            pnl_bps = direction * (-(y1 - y0) * 100)
        elif trade_type == 'spread_hedged':
            s0 = sp_pe[i]
            s1 = sp_pe[j]
            if np.isnan(s0) or np.isnan(s1):
                i += hp
                continue
            pnl_bps = direction * (-(s1 - s0))
        else:  # pure_residual
            r0 = resid_arr[i]
            r1 = resid_arr[j]
            if np.isnan(r0) or np.isnan(r1):
                i += hp
                continue
            pnl_bps = direction * (-(r1 - r0))
        trades.append({'entry_i': i, 'exit_i': j, 'direction': direction,
                       'pnl_bps': pnl_bps, 'z_entry': z})
        i += hp
    return trades

def compute_metrics(trades, trade_type, pca_w, z_w, hp):
    if len(trades) < 3:
        return None
    pnls = np.array([t['pnl_bps'] for t in trades])
    sp_pe = df_spreads_aligned['PE'].values
    yld_pe = yield_pe_raw

    # vol in bps from relative changes
    if trade_type == 'outright':
        diffs = np.diff(yld_pe * 100)
        levels = (yld_pe[:-1] * 100)
        mask = levels > 0
        rel = diffs[mask] / levels[mask]
        vol_bps = np.nanstd(rel) * np.nanmean(levels[mask])
    elif trade_type == 'spread_hedged':
        diffs = np.diff(sp_pe)
        levels = sp_pe[:-1]
        mask = levels > 0
        rel = diffs[mask] / levels[mask]
        vol_bps = np.nanstd(rel) * np.nanmean(levels[mask])
    else:
        resid_arr = residuals_store[pca_w]
        diffs = np.diff(resid_arr)
        levels = sp_pe[:-1]
        mask = levels > 0
        rel = diffs[mask] / levels[mask]
        vol_bps = np.nanstd(rel) * np.nanmean(levels[mask])

    if vol_bps == 0 or np.isnan(vol_bps):
        return None

    cum_pnl = np.cumsum(pnls)
    roll_max = np.maximum.accumulate(cum_pnl)
    max_dd = np.min(cum_pnl - roll_max)

    resid_arr = residuals_store[pca_w]
    conv_days = []
    for t in trades:
        r0 = resid_arr[t['entry_i']]
        target = r0 / 2
        for k in range(t['entry_i'], min(t['entry_i'] + 60, n)):
            if abs(resid_arr[k] - 0) <= abs(target):
                conv_days.append(k - t['entry_i'])
                break

    sharpe = (np.mean(pnls) / vol_bps) * np.sqrt(252 / hp)

    return {
        'pca_window': pca_w, 'zscore_window': z_w, 'holding_period': hp,
        'trade_type': trade_type,
        'n_trades': len(pnls),
        'n_cheap': sum(1 for t in trades if t['direction'] == 1),
        'n_rich': sum(1 for t in trades if t['direction'] == -1),
        'hit_rate': np.mean(pnls > 0),
        'avg_pnl_bps': np.mean(pnls),
        'median_pnl_bps': np.median(pnls),
        'total_pnl_bps': np.sum(pnls),
        'vol_bps': vol_bps,
        'sharpe': sharpe,
        'max_dd_bps': max_dd,
        'avg_convergence_days': np.mean(conv_days) if conv_days else np.nan,
        'worst_trade_bps': np.min(pnls),
        'best_trade_bps': np.max(pnls)
    }

# --- Run all 300 combinations ---
results = []
all_trades_store = {}
for pca_w, z_w, hp, tt in product(pca_windows, zscore_windows, holding_periods, trade_types):
    trades = run_backtest(pca_w, z_w, hp, tt)
    all_trades_store[(pca_w, z_w, hp, tt)] = trades
    m = compute_metrics(trades, tt, pca_w, z_w, hp)
    if m:
        results.append(m)

df_backtest_summary = pd.DataFrame(results).sort_values('sharpe', ascending=False).reset_index(drop=True)

# --- Top 5 highlight ---
top5 = df_backtest_summary.head(5)
print('Top 5 configurations by Sharpe:')
print(top5[['pca_window', 'zscore_window', 'holding_period', 'trade_type',
            'n_trades', 'hit_rate', 'avg_pnl_bps', 'vol_bps', 'sharpe', 'max_dd_bps']].to_string(index=False))

# --- Best config ---
best = df_backtest_summary.iloc[0]
best_pca_w = best['pca_window']
best_z_w = int(best['zscore_window'])
best_hp = int(best['holding_period'])
best_tt = best['trade_type']

# ============================================================
# HEATMAPS — one per pca_window per trade_type
# ============================================================
for tt in trade_types:
    fig, axes = plt.subplots(1, len(pca_windows), figsize=(20, 4), sharey=True)
    fig.suptitle(f'Sharpe heatmap — {tt}', fontsize=13)
    for ax, pw in zip(axes, pca_windows):
        sub = df_backtest_summary[(df_backtest_summary['trade_type'] == tt) &
                                   (df_backtest_summary['pca_window'].apply(lambda x: x == pw))]
        piv = sub.pivot_table(index='zscore_window', columns='holding_period', values='sharpe')
        sns.heatmap(piv, ax=ax, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
                    cbar=False, linewidths=0.5)
        ax.set_title(f'PCA window: {pw if pw else "full"}')
    plt.tight_layout()
    plt.show()

# ============================================================
# PC VISUALIZATIONS — full-sample
# ============================================================
# Scree plot
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(1, 6), evr_full * 100, color='steelblue')
ax.set_xlabel('PC')
ax.set_ylabel('Variance explained (%)')
ax.set_title('Scree plot — full-sample PCA')
for i, v in enumerate(evr_full):
    ax.text(i + 1, v * 100 + 0.5, f'{v*100:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

# Country loadings bar chart (PC1, PC2, PC3)
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(countries))
w = 0.25
for pi, (pc_idx, label) in enumerate(zip([0, 1, 2], ['PC1', 'PC2', 'PC3'])):
    ax.bar(x + pi * w, loadings_full[:, pc_idx], width=w, label=label)
ax.set_xticks(x + w)
ax.set_xticklabels(countries)
ax.legend()
ax.set_title('Country loadings on PC1–PC3 (full-sample)')
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

# PC1/PC2/PC3 scores time series
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for ax, scores, pc_i in zip(axes,
                             [pc1_scores_full, pc2_scores_full, pc3_scores_full],
                             [0, 1, 2]):
    ax.plot(idx, scores, linewidth=0.8)
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_title(f'PC{pc_i+1} — {evr_full[pc_i]*100:.1f}% variance explained')
plt.tight_layout()
plt.show()

# PC1 vs spread changes scatter — one per country
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=False)
for ax, c in zip(axes, countries):
    x_vals = pc1_scores_full
    y_vals = df_delta[c].values
    mask = ~(np.isnan(x_vals) | np.isnan(y_vals))
    ax.scatter(x_vals[mask], y_vals[mask], alpha=0.3, s=5)
    m_, b_ = np.polyfit(x_vals[mask], y_vals[mask], 1)
    ax.plot(np.sort(x_vals[mask]), m_ * np.sort(x_vals[mask]) + b_, 'r-', linewidth=1.5)
    r2 = np.corrcoef(x_vals[mask], y_vals[mask])[0, 1] ** 2
    ax.set_title(f'{c}  R²={r2:.2f}')
    ax.set_xlabel('PC1 score')
    ax.set_ylabel('Δspread (bps)')
plt.tight_layout()
plt.show()

# Rolling loadings — Peru PC1 loading across rolling windows
fig, ax = plt.subplots(figsize=(14, 5))
for pw in [w for w in pca_windows if w is not None]:
    l_arr = loadings_store[pw]
    if l_arr.ndim == 2:
        ax.plot(idx, l_arr[:, countries.index('PE')], label=f'PE (w={pw})', linewidth=1)
ax.axhline(0, color='black', linewidth=0.6)
ax.set_title('Peru PC1 loading — rolling windows')
ax.legend()
plt.tight_layout()
plt.show()

# ============================================================
# BEST-CONFIG DIAGNOSTIC PLOTS
# ============================================================
resid_best = residuals_store[best_pca_w]
z_best = zscore_store[best_pca_w][best_z_w]
resid_s = pd.Series(resid_best, index=idx)
z_s = pd.Series(z_best, index=idx)
sigma = resid_s.rolling(best_z_w, min_periods=best_z_w).std()

# Residual time series with ±1.5σ
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(idx, resid_s, linewidth=0.9, label='Residual PE')
ax.fill_between(idx, -1.5 * sigma, 1.5 * sigma, alpha=0.15, color='gray', label='±1.5σ')
ax.axhline(0, color='black', linewidth=0.6)
ax.set_title(f'Peru 10Y residual — PCA:{best_pca_w} Z:{best_z_w}d')
ax.legend()
plt.tight_layout()
plt.show()

# Z-score time series with shaded signal periods
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(idx, z_s, linewidth=0.9, color='navy')
ax.axhline(z_threshold, color='red', linewidth=1, linestyle='--', label=f'+{z_threshold}')
ax.axhline(-z_threshold, color='green', linewidth=1, linestyle='--', label=f'-{z_threshold}')
ax.fill_between(idx, z_threshold, z_s.where(z_s > z_threshold), alpha=0.25, color='red')
ax.fill_between(idx, -z_threshold, z_s.where(z_s < -z_threshold), alpha=0.25, color='green')
ax.set_title('Z-score — signal-active periods shaded')
ax.legend()
plt.tight_layout()
plt.show()

# Cumulative P&L — all three trade types (best config)
fig, ax = plt.subplots(figsize=(14, 5))
colors = {'outright': 'navy', 'spread_hedged': 'darkorange', 'pure_residual': 'green'}
for tt in trade_types:
    trd = all_trades_store[(best_pca_w, best_z_w, best_hp, tt)]
    if not trd:
        continue
    pnls_arr = np.array([t['pnl_bps'] for t in trd])
    dates_arr = [idx[t['exit_i']] for t in trd]
    cum = np.cumsum(pnls_arr)
    ax.plot(dates_arr, cum, label=tt, color=colors[tt], linewidth=1.5)
ax.axhline(0, color='black', linewidth=0.6)
ax.set_title(f'Cumulative P&L (bps) — PCA:{best_pca_w} Z:{best_z_w}d HP:{best_hp}d')
ax.legend()
plt.tight_layout()
plt.show()

# ============================================================
# CURRENT-DAY RV DASHBOARD
# ============================================================
print('\nRV Dashboard — as of', idx[-1].date())
print(df_rv_today.to_string(index=False))

# Horizontal bar chart — residuals
fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = ['red' if r > 0 else 'green' for r in df_rv_today['Residual (bps)']]
bars = ax.barh(df_rv_today['Country'], df_rv_today['Residual (bps)'], color=bar_colors)
for bar, z in zip(bars, df_rv_today['Z-score']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'z={z:.2f}', va='center', fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Current residual by country (bps)')
ax.set_xlabel('Residual (bps)')
plt.tight_layout()
plt.show()

# Spread vs fitted scatter (cross-sectional, current day)
fig, ax = plt.subplots(figsize=(6, 6))
fitted_vals = df_rv_today['Fitted (bps)'].values
actual_vals = df_rv_today['Spread (bps)'].values
ax.scatter(fitted_vals, actual_vals, s=80, zorder=5)
for _, row in df_rv_today.iterrows():
    ax.annotate(row['Country'], (row['Fitted (bps)'], row['Spread (bps)']),
                xytext=(5, 5), textcoords='offset points', fontsize=10)
mn = min(fitted_vals.min(), actual_vals.min()) * 0.95
mx = max(fitted_vals.max(), actual_vals.max()) * 1.05
ax.plot([mn, mx], [mn, mx], 'k--', linewidth=1, label='Fair value')
ax.set_xlabel('Fitted spread (bps)')
ax.set_ylabel('Actual spread (bps)')
ax.set_title('Spread vs fitted — current day')
ax.legend()
plt.tight_layout()
plt.show()

# Residual history last 252d per country
lookback = 252
fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)
for ax, c in zip(axes, countries):
    r_arr = country_residuals[c]
    r_s = pd.Series(r_arr, index=idx).iloc[-lookback:]
    s_bands = r_s.rolling(best_z_w, min_periods=1).std()
    ax.plot(r_s.index, r_s.values, linewidth=0.9)
    ax.fill_between(r_s.index, -1.5 * s_bands, 1.5 * s_bands, alpha=0.15, color='gray')
    ax.axhline(r_arr[-1], color='red', linewidth=1, linestyle='--', label='Today')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'{c} residual — last {lookback}d')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ============================================================
# DECAY PROFILE — best config
# ============================================================
horizon_list = [5, 10, 20, 40, 60]
decay_cheap = []
decay_rich = []
best_trades = all_trades_store[(best_pca_w, best_z_w, best_hp, best_tt)]
resid_best_arr = residuals_store[best_pca_w]

for h in horizon_list:
    ch_fwd = []
    ri_fwd = []
    for t in best_trades:
        ei = t['entry_i']
        hi = min(ei + h, n - 1)
        dr = resid_best_arr[hi] - resid_best_arr[ei]
        if np.isnan(dr):
            continue
        if t['direction'] == 1:
            ch_fwd.append(dr)
        else:
            ri_fwd.append(dr)
    decay_cheap.append(np.mean(ch_fwd) if ch_fwd else np.nan)
    decay_rich.append(np.mean(ri_fwd) if ri_fwd else np.nan)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(horizon_list, decay_cheap, marker='o', label='Cheap signal (expect neg)')
ax.plot(horizon_list, decay_rich, marker='s', label='Rich signal (expect pos)')
ax.axhline(0, color='black', linewidth=0.6)
ax.set_xlabel('Horizon (trading days)')
ax.set_ylabel('Avg forward Δresidual (bps)')
ax.set_title('Decay profile — best config')
ax.legend()
plt.tight_layout()
plt.show()

# ============================================================
# REGIME ANALYSIS — residual > ±3σ, no reversion within 40d
# ============================================================
sigma_full = pd.Series(resid_best, index=idx).rolling(252, min_periods=60).std()
z_3 = pd.Series(resid_best, index=idx) / sigma_full
regime_events = []
skip_until = 0
for i, (dt, z_val) in enumerate(z_3.items()):
    if i < skip_until:
        continue
    if abs(z_val) > 3:
        reverted = False
        sign0 = np.sign(z_val)
        for j in range(i + 1, min(i + 41, n)):
            if np.sign(z_3.iloc[j]) != sign0 or abs(z_3.iloc[j]) < 1.5:
                reverted = True
                break
        if not reverted:
            regime_events.append({'date': dt, 'z_score': round(z_val, 2),
                                   'residual_bps': round(resid_best[i], 1)})
            skip_until = i + 40

df_regime = pd.DataFrame(regime_events)
print('\nRegime breaks (|z|>3, no reversion in 40d):')
print(df_regime.to_string(index=False) if len(df_regime) else 'None found.')

print('\ndf_backtest_summary shape:', df_backtest_summary.shape)
print('df_rv_today:\n', df_rv_today.to_string(index=False))